# Notebook 04 — Preprocessing Pipeline Design

> **Design Document** — This notebook is a software-engineering design document, not an execution notebook. No code is run here. Every cell is a specification, rationale, or decision record that drives the implementation in the next stage.

---

## 1. Introduction

The previous notebooks established a solid empirical foundation for this project:

- **Notebook 01** explored the BraTS2020 dataset structure, NIfTI format, MRI modalities, and segmentation label definitions.
- **Notebook 02** validated dataset integrity, confirming consistent shapes (240 × 240 × 155), correct file naming, and perfect mask-to-volume alignment across all 369 patients.
- **Notebook 03** performed a medical data analysis that quantified class imbalance, tumor volume variability, slice ratios, spatial distributions, and bounding box statistics.
- **Notebook 03.5** defined the patient-level dataset split (Train / Validation / Test) and persisted the split as `dataset_split.json` for reproducibility.

This notebook — **Notebook 04** — synthesizes those findings into a complete, implementation-ready preprocessing pipeline design. It answers:

1. What does the raw input look like?
2. What does the model-ready output look like?
3. What transformations are needed, and in what order?
4. How does the pipeline differ between training, validation, and inference?
5. How is everything expressed in MONAI?

The outputs of this document directly drive **Notebook 05** (implementation).

## 2. Pipeline Overview

The preprocessing pipeline converts raw NIfTI MRI volumes into normalized, patch-sampled tensors ready for 3D segmentation training with MONAI and PyTorch.

The pipeline has three variants that share a common core but differ in augmentation and sampling strategy:

| Variant | Purpose | Augmentation | Patch Sampling |
|---------|---------|-------------|----------------|
| **Training** | Fit model weights | Yes (random spatial + intensity) | Foreground-aware random |
| **Validation** | Monitor generalization | No | Sliding window (overlap=0.5) |
| **Inference** | Generate predictions | No | Sliding window (overlap=0.5) |

All three variants share:
- Loading all four MRI modality paths as a list under a single `"image"` key
- Ensuring consistent channel-first format for both image and label simultaneously
- Label remapping (label 4 → label 3) for contiguous class indices
- Foreground-based spatial cropping (source varies: label for train/val, image for inference)
- Per-modality Z-score normalization (non-zero voxels only)
- Conversion to PyTorch `float32` tensors

## 3. Input Format

### 3.1 Raw Files on Disk

Each patient directory contains exactly five `.nii` files:

```
BraTS20_Training_001/
├── BraTS20_Training_001_flair.nii
├── BraTS20_Training_001_t1.nii
├── BraTS20_Training_001_t1ce.nii
├── BraTS20_Training_001_t2.nii
└── BraTS20_Training_001_seg.nii
```

### 3.2 Volume Properties (validated in Notebook 02)

| Property | Value |
|----------|-------|
| Spatial shape | 240 × 240 × 155 |
| Number of modalities | 4 (FLAIR, T1, T1ce, T2) |
| Segmentation labels | 0, 1, 2, 4 |
| Voxel spacing | Already standardized — no resampling required |
| File format | NIfTI (.nii) |
| Data type | float32 (MRI volumes), int (mask) |

### 3.3 Data Dictionary (MONAI dict format)

MONAI transforms operate on Python dictionaries. Each sample passed to the pipeline has the following structure:

```python
sample = {
    "image": [
        flair_path,   # channel 0
        t1_path,      # channel 1
        t1ce_path,    # channel 2
        t2_path,      # channel 3
    ],
    "label": seg_path,
}
```

`LoadImaged` reads all four file paths listed under `"image"` and concatenates them into a single tensor of shape **(4, H, W, D)**. The list order defines the channel order and must remain fixed throughout the project.

> **Channel order contract:** `[FLAIR, T1, T1ce, T2]` — this order must be identical in `dataset.py`, `train.py`, and `predict.py`.

## 4. Desired Output Format

After the full pipeline, each sample must conform to the following specification:

### 4.1 Image Tensor

| Property | Value |
|----------|-------|
| Shape | **(4, 128, 128, 128)** — four-channel 3D patch |
| Data type | `torch.float32` |
| Value range | Approximately −3 to +3 (Z-score normalized) |
| Channel order | FLAIR, T1, T1ce, T2 |

### 4.2 Label Tensor

| Property | Value |
|----------|-------|
| Shape | **(1, 128, 128, 128)** |
| Data type | `torch.int64` |
| Classes | 0 (background), 1 (NCR/NET), 2 (ED), 3 (ET) |

> **On One-Hot encoding:** The label remains as class indices (`int64`) throughout preprocessing. If one-hot representation is needed, it is applied inside the loss function or via a dedicated transform at training time — not in the core preprocessing pipeline.

> **Why label 4 → 3?** The original BraTS2020 masks use class 4 for Enhancing Tumor (ET), skipping class 3. MONAI and most deep learning frameworks expect contiguous class indices. Remapping label 4 → 3 avoids a wasted output channel and simplifies loss computation.

### 4.3 Batch Shape (as seen by the model)

```
image batch : (B, 4, 128, 128, 128)
label batch : (B, 1, 128, 128, 128)
```

## 5. Preprocessing Stages

The following table enumerates every preprocessing stage in execution order, with its rationale and empirical basis.

| # | Stage | Applied To | Rationale |
|---|-------|-----------|-----------|
| 1 | **LoadImaged** | image (list of 4 paths) + label | Read NIfTI files; concatenate 4 modalities → (4, H, W, D) |
| 2 | **EnsureChannelFirstd** | image + label | Normalize both to (C, H, W, D) simultaneously before any spatial op |
| 3 | **EnsureTyped** | image + label | Cast to float32 / int64 before all downstream transforms |
| 4 | **MapLabelValued** | label | Remap label 4 → 3 for contiguous class indices |
| 5 | **CropForegroundd** | image + label | Remove empty background borders; source key = `label` (train/val) or `image` (inference) |
| 6 | **NormalizeIntensityd** | image | Per-channel Z-score using non-zero voxels only |
| 7 | **Spatial augmentation** *(train only)* | image + label | Simulate acquisition variability |
| 8 | **Intensity augmentation** *(train only)* | image | Simulate scanner differences |
| 9 | **RandCropByPosNegLabeld** *(train)* / **SlidingWindowInferer** *(val/infer)* | image + label | Training: foreground-aware patch; Val/Infer: full-volume reconstruction |

---

### 5.1 Why This Ordering?

**EnsureChannelFirstd before EnsureTyped:**
Applying channel-first formatting to both `image` and `label` simultaneously ensures both tensors are in identical (C, H, W, D) format before any type casting or spatial operation. This matches the official MONAI examples and eliminates format inconsistencies between the two keys.

**EnsureTyped before MapLabelValued:**
Label remapping operates on numeric values. Ensuring the tensor type is fixed before remapping avoids subtle dtype-dependent edge cases.

**CropForeground before Normalization:**
If normalization were applied before cropping, the large background region (zero voxels) would distort the intensity statistics even if the `nonzero=True` flag is used — the compute overhead is unnecessary and the statistics would reflect a different spatial support than the final cropped volume. Cropping first guarantees normalization statistics are computed exclusively over the brain region that the model will actually see.

**Normalization before Augmentation:**
Intensity augmentation (`RandScaleIntensity`, `RandShiftIntensity`) is applied on top of normalized values, giving augmentation parameters a consistent, interpretable scale across all patients and modalities.

## 6. Transform Order

```
RAW NIfTI FILES
      │
      ▼
┌─────────────────────────────────────────────────────────┐
│  SHARED CORE  (all three pipeline variants)             │
│                                                         │
│  1. LoadImaged(keys=["image", "label"])                 │
│     └─ image: list of 4 paths → (4, H, W, D)           │
│     └─ label: single path    → (H, W, D)               │
│                                                         │
│  2. EnsureChannelFirstd(keys=["image", "label"])        │
│     └─ image: (4, H, W, D)  [already channeled]        │
│     └─ label: (1, H, W, D)                             │
│                                                         │
│  3. EnsureTyped(keys=["image", "label"])                │
│     └─ image → float32                                  │
│     └─ label → int64                                    │
│                                                         │
│  4. MapLabelValued(label: 4 → 3)                        │
│                                                         │
│  5. CropForegroundd(source_key="label")                 │
│     └─ train / val only                                 │
│     └─ inference: source_key="image"                    │
│                                                         │
│  6. NormalizeIntensityd(nonzero=True, channel_wise=True)│
└──────────────────┬──────────────────────────────────────┘
                   │
       ┌───────────┼────────────────┐
       ▼           ▼                ▼
  TRAINING     VALIDATION       INFERENCE
  ─────────    ──────────       ─────────
  RandFlipd    (no aug)         (no aug)
  RandRotate90d                  
  RandGaussianNoised             
  RandScaleIntensityd            
  RandShiftIntensityd            
       │           │                │
  RandCropBy   SlidingWindow   SlidingWindow
  PosNegLabeld  Inferer         Inferer
  (128³)       (128³, ov=0.5)  (128³, ov=0.5)
```

## 7. MONAI Pipeline Design

This section specifies the MONAI transform composition for each pipeline variant using the dictionary-based API (`monai.transforms.Compose`).

All transforms listed below are design specifications. Implementation lives in Notebook 05.

### 7.1 Key Naming Convention

| Key | Content |
|-----|---------|
| `"image"` | 4-channel MRI tensor — FLAIR, T1, T1ce, T2 |
| `"label"` | Segmentation mask tensor (absent during inference) |
| `"pred"` | Model output tensor (inference post-processing only) |

### 7.2 Shared Core Transforms

```python
shared_transforms = [

    # 1. Load all four modality files + segmentation mask
    #    image key receives a list of 4 paths → output shape: (4, H, W, D)
    LoadImaged(
        keys=["image", "label"],
        image_only=False,
    ),

    # 2. Normalize channel dimension for BOTH image and label simultaneously
    #    image: already (4, H, W, D) after multi-path load
    #    label: (H, W, D) → (1, H, W, D)
    EnsureChannelFirstd(keys=["image", "label"]),

    # 3. Cast to correct dtypes before all downstream operations
    EnsureTyped(
        keys=["image", "label"],
        dtype=[torch.float32, torch.int64],
    ),

    # 4. Remap label 4 → 3 for contiguous class indices [0, 1, 2, 3]
    MapLabelValued(
        keys="label",
        orig_labels=[0, 1, 2, 4],
        target_labels=[0, 1, 2, 3],
    ),

    # 5. Remove empty background borders
    #    source_key="label": crop is guided by non-zero mask voxels
    #    ⚠️ Inference variant uses source_key="image" — see Section 10
    CropForegroundd(
        keys=["image", "label"],
        source_key="label",
    ),

    # 6. Per-channel Z-score normalization (non-zero brain voxels only)
    NormalizeIntensityd(
        keys="image",
        nonzero=True,
        channel_wise=True,
    ),
]
```

## 8. Training Pipeline

The training pipeline appends spatial and intensity augmentation transforms followed by foreground-aware random patch sampling.

```python
train_transforms = Compose(
    shared_transforms + [

        # ── Spatial Augmentation ──────────────────────────────────────────

        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),

        RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),

        # ── Intensity Augmentation ────────────────────────────────────────

        RandGaussianNoised(keys="image", prob=0.15, std=0.01),
        RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),
        RandShiftIntensityd(keys="image", offsets=0.1, prob=0.5),

        # ── Foreground-Aware Patch Sampling ───────────────────────────────

        RandCropByPosNegLabeld(
            keys=["image", "label"],
            label_key="label",
            spatial_size=(128, 128, 128),
            pos=1,          # 50% of patches centered on foreground voxel
            neg=1,          # 50% of patches centered on background voxel
            num_samples=2,  # patches extracted per volume per iteration
        ),
    ]
)
```

### 8.1 Augmentation Rationale

| Transform | Probability | Why |
|-----------|-------------|-----|
| `RandFlipd` (×3 axes) | 0.5 | Brain bilateral symmetry; arbitrary acquisition orientation |
| `RandRotate90d` | 0.5 | Scanner bed orientation variability across institutions |
| `RandGaussianNoised` | 0.15 | MRI thermal noise; low probability to preserve signal integrity |
| `RandScaleIntensityd` | 0.5 | Scanner gain differences across sites and field strengths |
| `RandShiftIntensityd` | 0.5 | Intensity bias field variation inherent in MRI acquisition |

### 8.2 Patch Size Justification

**Chosen size: 128 × 128 × 128**

| Candidate | Verdict | Reason |
|-----------|---------|--------|
| **96³** | ✗ Too small | May clip large tumors (max bounding box depth > 100 slices in some patients) |
| **128³** | ✓ Selected | Comfortably exceeds mean bounding box (~63 × 84 × 67); fits in 16–40 GB GPU at batch size 2 |
| **160³** | ✗ Too large | No meaningful coverage gain per Notebook 03 bounding box statistics; ~2× memory cost vs 128³ |

**Foreground-aware sampling rationale (`pos=1, neg=1`):**
Tumor voxels represent only ~1.11% of all voxels (Notebook 03). Purely random patch sampling would yield almost entirely background patches, making training inefficient. Setting `pos=1, neg=1` guarantees that 50% of sampled patches are anchored on a tumor voxel, directly addressing the class imbalance at the sampling level.

## 9. Validation Pipeline

The validation pipeline applies the shared core transforms only — no augmentation. Predictions are generated over the full volume using a sliding window inferer.

```python
val_transforms = Compose(shared_transforms)
# No augmentation transforms.
# No patch sampling — full cropped volume is passed to SlidingWindowInferer.
```

### 9.1 Sliding Window Inference (Validation)

```python
from monai.inferers import SlidingWindowInferer

val_inferer = SlidingWindowInferer(
    roi_size=(128, 128, 128),
    sw_batch_size=2,
    overlap=0.5,         # 50% overlap — reduces boundary artifacts
    mode="gaussian",     # Gaussian weighting further reduces seam artifacts
)
```

### 9.2 Validation Metric

Dice score computed per BraTS subregion and averaged:

| Metric | Target Labels |
|--------|--------------|
| Whole Tumor (WT) | 1 + 2 + 3 |
| Tumor Core (TC) | 1 + 3 |
| Enhancing Tumor (ET) | 3 |
| **Mean Dice** | Average of WT, TC, ET |

Mean Dice is the primary model selection criterion (checkpoint saving trigger).

## 10. Inference Pipeline

The inference pipeline omits the `"label"` key entirely, as ground-truth masks are unavailable for test patients.

> ⚠️ **Key difference from training/validation:** `CropForegroundd` uses `source_key="image"` instead of `source_key="label"`, since there is no mask available to guide the crop.

```python
infer_transforms = Compose([

    LoadImaged(keys="image", image_only=False),

    EnsureChannelFirstd(keys="image"),

    EnsureTyped(keys="image", dtype=torch.float32),

    # Crop guided by image content (non-zero brain voxels)
    CropForegroundd(
        keys="image",
        source_key="image",
    ),

    NormalizeIntensityd(
        keys="image",
        nonzero=True,
        channel_wise=True,
    ),
])
```

### 10.1 Post-Processing

After the sliding window inferer generates a raw logit volume:

```python
post_transforms = Compose([

    # Convert network logits → class probabilities → class indices
    Activationsd(keys="pred", softmax=True),
    AsDiscreted(keys="pred", argmax=True),

    # Restore original BraTS label convention (3 → 4) for submission format
    MapLabelValued(
        keys="pred",
        orig_labels=[0, 1, 2, 3],
        target_labels=[0, 1, 2, 4],
    ),

    # Save prediction as NIfTI
    SaveImaged(
        keys="pred",
        output_dir="/path/to/predictions",
        output_postfix="seg",
    ),
])
```

> **Label restoration:** Predictions are remapped back from [0, 1, 2, 3] → [0, 1, 2, 4] to match the BraTS2020 submission format before saving.

## 11. Assumptions

This pipeline design is built on the following assumptions, all validated in prior notebooks:

| Assumption | Source |
|-----------|--------|
| All MRI volumes have identical spatial dimensions (240 × 240 × 155) | Notebook 02 — Shape validation |
| All patients contain exactly four MRI modalities (FLAIR, T1, T1ce, T2) | Notebook 02 — File naming validation |
| Voxel spacing is already standardized — no resampling required | Notebook 02 — Shape validation |
| Dataset split is fixed and stored in `dataset_split.json` | Notebook 03.5 — Dataset split |
| Segmentation labels are always a subset of {0, 1, 2, 4} | Notebook 02 — Label validation |
| One patient (`BraTS20_Training_355`) has a non-standard seg filename — pipeline handles it via keyword matching | Notebook 02 — Naming edge case |

If any of these assumptions are violated (e.g., a new patient folder with different dimensions), the pipeline will raise an explicit error before any model training begins.

## 12. Final Engineering Decisions

### 12.1 Decision Table

| Decision | Rationale | Source |
|----------|-----------|--------|
| All 4 modalities as input channels | Each modality provides complementary tissue contrast; no single modality is sufficient | Notebook 03 — Modality comparison |
| No spatial resampling | All 369 patients share identical spatial dimensions | Notebook 02 — Shape validation |
| Label 4 → 3 remapping | Contiguous class indices required by MONAI loss functions | Standard BraTS preprocessing |
| EnsureChannelFirstd applied to image + label together | Consistent format from the start; matches official MONAI examples | MONAI best practices |
| EnsureTyped before MapLabelValued | Fixed dtype avoids edge cases in label arithmetic | Engineering judgment |
| CropForeground before normalization | Normalization statistics reflect only brain tissue, not background | Notebook 03 — Slice ratio analysis |
| Per-channel Z-score (nonzero mask) | Different modalities have incompatible intensity scales | Notebook 01 — Intensity inspection |
| Patch size: 128 × 128 × 128 | Exceeds mean tumor bounding box; fits in GPU at batch 2; 96³ too small, 160³ no gain | Notebook 03 — Bounding box analysis |
| Foreground-aware sampling (pos=neg=1) | Tumor occupies only 1.11% of voxels; random sampling is inefficient | Notebook 03 — Class imbalance |
| label → int64, no one-hot in preprocessing | One-hot handled inside loss function; keeps preprocessing simple and unambiguous | Engineering judgment |
| CropForeground source: label (train/val), image (infer) | No mask available at inference time | Design requirement |
| Sliding window overlap=0.5, mode=gaussian | Reduces boundary seam artifacts in full-volume prediction | Standard MONAI practice |
| Dice-based loss | Cross-entropy ineffective at 98.89% background | Notebook 03 — Voxel distribution |
| Mean Dice (WT, TC, ET) as validation metric | Standard BraTS challenge evaluation protocol | BraTS challenge specification |

---

### 12.2 Final ROI Size — Canonical Decision

> This is the single authoritative value for patch size across the entire project. All downstream files (`preprocessing.py`, `dataset.py`, `train.py`, `predict.py`) must use this exact value.

| Parameter | Value |
|-----------|-------|
| **ROI Size** | **128 × 128 × 128** |
| Sliding window `roi_size` | (128, 128, 128) |
| `RandCropByPosNegLabeld` `spatial_size` | (128, 128, 128) |
| `SlidingWindowInferer` `roi_size` | (128, 128, 128) |

---

## Summary

This design document fully specifies the preprocessing pipeline for the BraTS2020 brain tumor segmentation project. Every decision is grounded in empirical observations from Notebooks 01–03.5 and expressed as unambiguous MONAI transform specifications.

**Next step: Notebook 05 — Preprocessing Pipeline Implementation**

The implementation will:
1. Translate every transform specified here into executable MONAI code
2. Build `Dataset` and `DataLoader` objects for training and validation
3. Validate pipeline output shapes and intensity distributions with assertions
4. Run a single forward pass to confirm end-to-end compatibility with the model input